# PCB X — YOLO training notebook (top + bottom models)

**Portable Edge-AI Inspection Chamber for Dual-Sided PCB Component Detection**

This notebook trains the two detectors (top side = through-hole parts, bottom side = SMD parts),
validates them, optionally scores the test set, exports them to **NCNN** for the Raspberry Pi 5,
and packages the files for download.

**Before you start**
1. *Runtime → Change runtime type → T4 GPU* (free tier is enough).
2. Prepare the dataset on your PC with `training/tools/prepare_dataset.py` (both sides), check it with
   `tools/check_dataset.py`, and zip it with `tools/make_dataset_zip.py`.
3. Upload `pcbx_dataset.zip` to Google Drive as **`MyDrive/PCBX/pcbx_dataset.zip`**.
4. Run the cells **in order** (*Runtime → Run all* works too). Only cell 2 has settings.

Training runs are saved **in Google Drive** (`MyDrive/PCBX/runs`), so a Colab disconnect does not lose
your work — re-run the notebook and the interrupted training resumes.

The same notebook also runs on a local PC (Jupyter / VS Code): it detects that it is not on Colab
and uses the `training/` folder instead of Drive.

In [ ]:
# 1) Environment check
import sys, platform, subprocess
IN_COLAB = "google.colab" in sys.modules
print("Running on Colab:", IN_COLAB, "| Python", platform.python_version())
try:
    print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"],
                         capture_output=True, text=True).stdout)
except FileNotFoundError:
    print("No NVIDIA GPU visible. On Colab: Runtime > Change runtime type > T4 GPU.")

In [ ]:
# 2) SETTINGS - the only cell you normally edit
from pathlib import Path

BASE_MODEL = "yolov8n.pt"      # baseline from the PDF. Try "yolo26n.pt" as a second experiment.
SIDES_TO_TRAIN = ["top", "bottom"]
RUN_NAME_SUFFIX = ""           # e.g. "_v2" to keep several experiments side by side
RUN_FINAL_TEST = False         # set True ONLY for the final Chapter 4 evaluation (run the test set once)
RESUME_IF_POSSIBLE = True      # continue an interrupted training run found in the runs folder
ULTRALYTICS_VERSION = "8.4.218"  # must equal the version installed on the Raspberry Pi

if IN_COLAB:
    PROJECT_DIR = Path("/content/drive/MyDrive/PCBX")      # Drive folder (persistent)
    WORK_DIR = Path("/content/work")                         # fast local disk (temporary)
else:
    PROJECT_DIR = Path.cwd()                                 # run this notebook from training/
    WORK_DIR = PROJECT_DIR
DATASET_ZIP = PROJECT_DIR / "pcbx_dataset.zip"
RUNS_DIR = PROJECT_DIR / "runs"
EXPORT_DIR = PROJECT_DIR / "export_out"

CLASSES = {
    "top": ["electrolytic_cap", "inductor", "transformer", "varistor",
            "fuse", "terminal_block", "connector"],
    "bottom": ["resistor", "smd_capacitor", "diode", "ic8"],
}
CANVAS_W, CANVAS_H = 1200, 500     # canvas made by the Pi software (10 px/mm)
print("Project folder:", PROJECT_DIR)

In [ ]:
# 3) Mount Google Drive (Colab only)
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset zip found:", DATASET_ZIP.exists(), DATASET_ZIP)
if not DATASET_ZIP.exists() and not (WORK_DIR / "datasets").exists():
    raise FileNotFoundError(f"Upload pcbx_dataset.zip to {DATASET_ZIP}")

In [ ]:
# 4) Install libraries (pinned version = same as on the Raspberry Pi)
%pip install -q "ultralytics==$ULTRALYTICS_VERSION" pyyaml ncnn pnnx
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__,
      "| CUDA:", torch.cuda.is_available())

In [ ]:
# 5) Extract the dataset to the fast local disk
import zipfile, shutil
DATASETS = WORK_DIR / "datasets"
if DATASET_ZIP.exists():
    if DATASETS.exists() and IN_COLAB:
        shutil.rmtree(DATASETS)
    DATASETS.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(DATASET_ZIP) as z:
        bad = [n for n in z.namelist() if "\\" in n]
        if bad:
            raise ValueError("Zip contains Windows backslash paths - make it with "
                             "tools/make_dataset_zip.py. Example: " + bad[0])
        z.extractall(DATASETS)

def find_dataset(side):
    hits = [p for p in DATASETS.rglob(f"pcbx_{side}") if (p / "images").is_dir()]
    return hits[0] if hits else None

DS = {s: find_dataset(s) for s in ("top", "bottom")}
for s, p in DS.items():
    print(s, "->", p)

In [ ]:
# 6) Verify dataset structure, labels and board-wise split
from collections import Counter, defaultdict
import re

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
def board_id(name):
    m = re.match(r"^([A-Za-z]+\d+)_", name)
    return m.group(1) if m else "UNKNOWN"

def verify(side, root):
    names, errors = CLASSES[side], []
    counts = {s: Counter() for s in ("train", "val", "test")}
    n_img, boards = Counter(), defaultdict(set)
    for s in ("train", "val", "test"):
        idir, ldir = root / "images" / s, root / "labels" / s
        if not idir.is_dir() or not ldir.is_dir():
            errors.append(f"missing {idir} or {ldir}"); continue
        for ip in idir.iterdir():
            if ip.suffix.lower() not in IMG_EXT: continue
            n_img[s] += 1; boards[board_id(ip.name)].add(s)
            lp = ldir / (ip.stem + ".txt")
            if not lp.exists():
                errors.append(f"no label for {s}/{ip.name}"); continue
            for k, line in enumerate(lp.read_text().splitlines(), 1):
                p = line.split()
                if not p: continue
                if len(p) != 5 or not (0 <= int(p[0]) < len(names)):
                    errors.append(f"bad line {lp.name}:{k}: {line}"); continue
                if not all(0 <= float(v) <= 1 for v in p[1:]):
                    errors.append(f"coords not 0..1 in {lp.name}:{k}")
                counts[s][names[int(p[0])]] += 1
    leaks = {b: sorted(v) for b, v in boards.items() if len(v) > 1}
    print(f"\n=== {side.upper()} ({root}) ===")
    print(f"{'class':18}{'train':>7}{'val':>7}{'test':>7}")
    for n in names:
        print(f"{n:18}" + "".join(f"{counts[s][n]:7d}" for s in ("train", "val", "test")))
    print(f"{'images':18}" + "".join(f"{n_img[s]:7d}" for s in ("train", "val", "test")),
          "| boards:", len(boards))
    if leaks: errors.append(f"boards in several splits (leakage): {leaks}")
    for n in names:
        if counts["val"][n] == 0: print(f"WARNING: no '{n}' in validation set")
    for e in errors[:20]: print("ERROR:", e)
    if errors: raise ValueError(f"{side}: {len(errors)} dataset errors - fix them first")
    print("OK")

for side in SIDES_TO_TRAIN:
    if DS[side] is None:
        raise FileNotFoundError(f"pcbx_{side} not found inside the zip")
    verify(side, DS[side])

In [ ]:
# 7) Write the dataset YAML files (absolute paths avoid the common 'images not found' error)
import yaml
DATA_YAML = {}
for side in ("top", "bottom"):
    if DS[side] is None: continue
    cfg = {"path": DS[side].resolve().as_posix(), "train": "images/train", "val": "images/val",
           "test": "images/test", "names": {i: n for i, n in enumerate(CLASSES[side])}}
    DATA_YAML[side] = WORK_DIR / f"pcbx_{side}.yaml"
    DATA_YAML[side].write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(DATA_YAML[side].read_text())

In [ ]:
# 8) Training settings - identical to training/configs/hyp_top.yaml and hyp_bottom.yaml
COMMON = dict(epochs=150, patience=30, optimizer="auto", seed=0, deterministic=True,
              cache=True, workers=2, plots=True,
              fliplr=0.0, flipud=0.0, degrees=3.0, translate=0.05, scale=0.2,
              shear=0.0, perspective=0.0, mosaic=1.0, close_mosaic=10, mixup=0.0,
              hsv_h=0.01, hsv_s=0.4, hsv_v=0.3)
HYP = {
    "top":    dict(COMMON, imgsz=640,  batch=16),   # large through-hole parts
    "bottom": dict(COMMON, imgsz=1216, batch=8),    # small SMD parts: keep full canvas resolution
}
import math
def rect_export_size(imgsz):
    s = imgsz / max(CANVAS_W, CANVAS_H)
    return [int(math.ceil(CANVAS_H * s / 32) * 32), int(math.ceil(CANVAS_W * s / 32) * 32)]
for s in HYP: print(s, HYP[s]["imgsz"], "-> export imgsz", rect_export_size(HYP[s]["imgsz"]))

In [ ]:
# 9) Training helper (resumes automatically after a Colab disconnect)
from ultralytics import YOLO

def run_name(side):
    return f"{side}_{Path(BASE_MODEL).stem}{RUN_NAME_SUFFIX}"

def train_side(side):
    name = run_name(side)
    run_dir = RUNS_DIR / side / name
    last, best = run_dir / "weights" / "last.pt", run_dir / "weights" / "best.pt"
    if RESUME_IF_POSSIBLE and last.exists():
        try:
            print("Found", last, "-> trying to resume")
            YOLO(str(last)).train(resume=True)
            return best
        except Exception as e:      # finished runs cannot be resumed
            print("Not resumed:", e)
            if best.exists():
                print("Using existing", best)
                return best
    model = YOLO(BASE_MODEL)
    model.train(data=str(DATA_YAML[side]), project=str(RUNS_DIR / side), name=name,
                exist_ok=True, device=0 if torch.cuda.is_available() else "cpu", **HYP[side])
    return best

BEST = {}

In [ ]:
# 10) Train the TOP-side model
if "top" in SIDES_TO_TRAIN:
    BEST["top"] = train_side("top")
    print("TOP best weights:", BEST["top"])

In [ ]:
# 11) Train the BOTTOM-side model (takes longer: 1216 px images)
if "bottom" in SIDES_TO_TRAIN:
    BEST["bottom"] = train_side("bottom")
    print("BOTTOM best weights:", BEST["bottom"])

In [ ]:
# 12) Validate on the VALIDATION set (use these numbers to make decisions)
import json
def report(side, split, weights, imgsz, conf=0.001):
    m = YOLO(str(weights), task="detect").val(data=str(DATA_YAML[side]), split=split, imgsz=imgsz,
                                              conf=conf, project=str(RUNS_DIR / "val"),
                                              name=f"{side}_{split}", exist_ok=True, plots=True,
                                              verbose=False)
    names = CLASSES[side]
    print(f"\n{side.upper()} - {split}   weights: {weights}")
    print(f"{'class':18}{'P':>7}{'R':>7}{'F1':>7}{'mAP50':>8}{'mAP50-95':>10}")
    rows = []
    for i, c in enumerate(m.box.ap_class_index):
        p, r = m.box.p[i], m.box.r[i]; f1 = 2 * p * r / (p + r + 1e-9)
        rows.append(dict(cls=names[int(c)], P=float(p), R=float(r), F1=float(f1),
                         mAP50=float(m.box.ap50[i]), mAP50_95=float(m.box.ap[i])))
        print(f"{names[int(c)]:18}{p:7.3f}{r:7.3f}{f1:7.3f}{m.box.ap50[i]:8.3f}{m.box.ap[i]:10.3f}")
    mp, mr = m.box.mp, m.box.mr
    print(f"{'ALL':18}{mp:7.3f}{mr:7.3f}{2*mp*mr/(mp+mr+1e-9):7.3f}{m.box.map50:8.3f}{m.box.map:10.3f}")
    out = RUNS_DIR / "val" / f"metrics_{side}_{split}.json"
    out.write_text(json.dumps(dict(side=side, split=split, weights=str(weights), imgsz=imgsz,
                                   mAP50=float(m.box.map50), mAP50_95=float(m.box.map),
                                   P=float(mp), R=float(mr), per_class=rows), indent=2))
    return m

for side in SIDES_TO_TRAIN:
    report(side, "val", BEST[side], HYP[side]["imgsz"])

In [ ]:
# 13) Look at the training curves and confusion matrices
from IPython.display import Image, display
for side in SIDES_TO_TRAIN:
    run_dir = RUNS_DIR / side / run_name(side)
    for f in ["results.png", "confusion_matrix_normalized.png", "BoxPR_curve.png", "val_batch0_pred.jpg"]:
        p = run_dir / f
        if p.exists():
            print(side, f); display(Image(filename=str(p), width=900))

In [ ]:
# 14) FINAL TEST - run ONCE, after every decision (model size, imgsz, thresholds) is frozen
if RUN_FINAL_TEST:
    for side in SIDES_TO_TRAIN:
        report(side, "test", BEST[side], HYP[side]["imgsz"])
    print("Test results saved in", RUNS_DIR / "val")
else:
    print("Skipped. Set RUN_FINAL_TEST = True in cell 2 for the final evaluation only.")

In [ ]:
# 15) Export to NCNN for the Raspberry Pi 5 and compare with the PyTorch model
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
EXPORTED = {}
for side in SIDES_TO_TRAIN:
    size = rect_export_size(HYP[side]["imgsz"])
    path = Path(YOLO(str(BEST[side])).export(format="ncnn", imgsz=size))
    target = EXPORT_DIR / f"{side}_best_ncnn_model"
    if target.exists(): shutil.rmtree(target)
    shutil.copytree(path, target)
    EXPORTED[side] = (target, size)
    print(f"{side}: {target}  imgsz={size}")
    # sanity check on 5 validation images
    pt, nc = YOLO(str(BEST[side])), YOLO(str(target), task="detect")
    for img in sorted((DS[side] / "images" / "val").glob("*"))[:5]:
        a = len(pt.predict(str(img), imgsz=HYP[side]["imgsz"], conf=0.25, verbose=False)[0].boxes)
        b = len(nc.predict(str(img), imgsz=size, conf=0.25, verbose=False)[0].boxes)
        print(f"   {img.name:40} pt={a:3d}  ncnn={b:3d}" + ("" if abs(a - b) <= 1 else "   <-- check"))
    # NCNN accuracy on validation (should be within ~1 mAP point of the .pt)
    report(side, "val", target, size)

In [ ]:
# 16) Package everything the Raspberry Pi needs + the evidence for your paper
from datetime import datetime
stamp = datetime.now().strftime("%Y%m%d_%H%M")
pkg = PROJECT_DIR / f"pcbx_models_{Path(BASE_MODEL).stem}_{stamp}.zip"
with zipfile.ZipFile(pkg, "w", zipfile.ZIP_DEFLATED) as z:
    for side, (target, size) in EXPORTED.items():
        for f in target.rglob("*"):
            z.write(f, f"models/{target.name}/{f.relative_to(target).as_posix()}")
        z.write(BEST[side], f"weights/{side}_best.pt")
        run_dir = RUNS_DIR / side / run_name(side)
        for f in ["results.csv", "results.png", "confusion_matrix.png",
                  "confusion_matrix_normalized.png", "args.yaml", "BoxPR_curve.png"]:
            if (run_dir / f).exists():
                z.write(run_dir / f, f"evidence/{side}/{f}")
    for f in (RUNS_DIR / "val").glob("metrics_*.json"):
        z.write(f, f"evidence/{f.name}")
    z.writestr("README.txt",
               "Copy models/* to pcbx_inspection/models/ on the Raspberry Pi.\n"
               + "".join(f"config.py MODELS['{s}']: path='{t.name}', imgsz={tuple(sz)}\n"
                         for s, (t, sz) in EXPORTED.items())
               + f"Trained with ultralytics {ultralytics.__version__}, base model {BASE_MODEL}\n")
print("Package:", pkg, f"{pkg.stat().st_size/1e6:.1f} MB")
if IN_COLAB:
    from google.colab import files
    files.download(str(pkg))     # also stays in your Drive folder

In [ ]:
# 17) OPTIONAL - learning-curve test (25/50/75/100 % of the training images)
#     Takes 4x the training time per side. Run after the main training.
RUN_LEARNING_CURVE = False
LC_SIDE, LC_EPOCHS = "bottom", 100
if RUN_LEARNING_CURVE:
    import random, csv
    ds = DS[LC_SIDE]
    imgs = sorted((ds / "images" / "train").glob("*"))
    random.Random(0).shuffle(imgs)
    base = yaml.safe_load(DATA_YAML[LC_SIDE].read_text())
    rows = []
    for frac in (0.25, 0.5, 0.75, 1.0):
        n = max(1, round(len(imgs) * frac))
        lst = WORK_DIR / f"lc_{LC_SIDE}_{int(frac*100)}.txt"
        lst.write_text("\n".join(p.as_posix() for p in imgs[:n]) + "\n")
        dy = WORK_DIR / f"lc_{LC_SIDE}_{int(frac*100)}.yaml"
        dy.write_text(yaml.safe_dump(dict(base, train=lst.as_posix()), sort_keys=False))
        name = f"lc_{int(frac*100)}"
        YOLO(BASE_MODEL).train(data=str(dy), project=str(RUNS_DIR / "learning_curve" / LC_SIDE),
                               name=name, exist_ok=True, **dict(HYP[LC_SIDE], epochs=LC_EPOCHS))
        w = RUNS_DIR / "learning_curve" / LC_SIDE / name / "weights" / "best.pt"
        m = YOLO(str(w)).val(data=str(dy), split="val", imgsz=HYP[LC_SIDE]["imgsz"], verbose=False, plots=False)
        rows.append(dict(fraction=frac, images=n, mAP50=round(float(m.box.map50), 4)))
        print(rows[-1])
    import matplotlib.pyplot as plt
    plt.plot([r["images"] for r in rows], [100 * r["mAP50"] for r in rows], "o-")
    plt.xlabel("training images"); plt.ylabel("val mAP@50 (%)"); plt.grid(alpha=.3)
    plt.title(f"Learning curve - {LC_SIDE}")
    plt.savefig(RUNS_DIR / "learning_curve" / f"{LC_SIDE}.png", dpi=150); plt.show()
    gain = 100 * (rows[-1]["mAP50"] - rows[-2]["mAP50"])
    print(f"Gain 75% -> 100%: {gain:+.1f} points ->", "capture more images" if gain > 2 else "enough data")

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `Dataset images not found` / `missing path` | relative `path:` in a YAML | cell 7 writes absolute paths — run it again after cell 5 |
| `FileNotFoundError: pcbx_top not found inside the zip` | zip made from the wrong folder | zip with `tools/make_dataset_zip.py` (contains `pcbx_top/` and `pcbx_bottom/` at the top) |
| file names like `images\train\B01.jpg` | zip made with old Windows tools | use `tools/make_dataset_zip.py` |
| `CUDA out of memory` | batch too large for 1216 px | cell 8: bottom `batch=4` |
| Colab disconnected mid-training | free-tier time limit | re-run all cells; cell 9 resumes from `last.pt` in Drive |
| mAP stays ~0 | labels wrong or class IDs swapped | open `_previews/` from `check_dataset.py`; check `classes.txt` remap printed by `prepare_dataset.py` |
| very high val mAP but bad on new boards | same board in train and val | split by board ID (the check in cell 6 reports leakage) |
| NCNN box count differs a lot from `.pt` | export size ≠ predict size | always predict with the printed export `imgsz`; set the same in `config.py` |
| `yolo26n.pt` not found | old ultralytics | keep `ULTRALYTICS_VERSION` ≥ 8.4 (YOLO26 support) |
| `ModuleNotFoundError: ncnn` during export | export deps missing | cell 4 installs `ncnn` and `pnnx`; re-run it |